# PyTorch：张量、自动微分与训练


由 [Vision Learning Hub](https://charlie-sys.github.io/vision-learning-hub/) 提供。
在 Colab 登录自己的Google账号，按从上到下的顺序运行；修改前可“在云端硬盘中保存副本”。
若需要GPU，在“运行时 → 更改运行时类型”中选择可用GPU。免费GPU不保证随时可用，断线后临时文件可能丢失，请下载结果。
本Notebook的结果属于你自己的运行，不应把教程实测数值直接当作本次成绩。


## 1. 库与设备
Colab预装框架；若在本地运行，先按PyTorch官方安装向导安装与硬件匹配的版本。


In [ ]:
import torch
import matplotlib.pyplot as plt
print('torch:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
torch.manual_seed(7)


## 2. 自动微分
例题预期loss=2，w的梯度=4。注意每次backward会累加梯度。


In [ ]:
w = torch.tensor(3., requires_grad=True, device=device)
x, b, target = 2., 1., 5.
loss = (w*x+b-target)**2/2
loss.backward()
print('loss:', loss.item(), 'gradient:', w.grad.item())
assert torch.isclose(w.grad, torch.tensor(4., device=device))


## 3. 完整线性回归
目标为y=3x+1，添加少量噪声。训练和验证数据独立生成。


In [ ]:
x_train = torch.randn(256, 1, device=device)
y_train = 3*x_train + 1 + .1*torch.randn_like(x_train)
x_val = torch.randn(128, 1, device=device)
y_val = 3*x_val + 1 + .1*torch.randn_like(x_val)
model = torch.nn.Linear(1, 1).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=.1)
history = []
for step in range(100):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x_train), y_train)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
model.eval()
with torch.inference_mode():
    val_loss = torch.nn.functional.mse_loss(model(x_val), y_val).item()
print('learned weight/bias:', model.weight.item(), model.bias.item())
print('validation MSE:', val_loss)
assert val_loss < .1
plt.plot(history)
plt.xlabel('optimization step')
plt.ylabel('training MSE')
plt.show()


## 4. 保存与恢复
训练输出应及时下载，运行时重置会丢失临时文件。


In [ ]:
torch.save(model.state_dict(), 'linear_torch.pt')
restored = torch.nn.Linear(1, 1).to(device)
restored.load_state_dict(torch.load('linear_torch.pt', map_location=device, weights_only=True))
with torch.inference_mode():
    torch.testing.assert_close(model(x_val), restored(x_val))
print('checkpoint reload passed')


## 练习
将学习率改为1，观察是否发散；与0.01比较收敛速度。不能用验证数据参与更新。
